# Baseline API Observability

This notebook verifies the running FastAPI server and gives a live overview of the Mini Wikipedia RAG system.

**Prerequisites**: start the server in a separate terminal before running these cells.
```
uv run uvicorn api:app --app-dir src --host 127.0.0.1 --port 8000
```

In [8]:
import requests
import json

BASE_URL = "http://127.0.0.1:8000"
print(f"Targeting server at: {BASE_URL}")

Targeting server at: http://127.0.0.1:8000


## 1. Health Check

Confirm the server is reachable and reports `ok`.

In [9]:
response = requests.get(f"{BASE_URL}/health")
print(f"Status code : {response.status_code}")
print(f"Payload     : {response.json()}")
assert response.json()["status"] == "ok", "Health check failed!"
print("✓ Server is healthy")

Status code : 200
Payload     : {'status': 'ok'}
✓ Server is healthy


## 2. Root & Version

In [10]:
for path in ["/", "/version"]:
    r = requests.get(f"{BASE_URL}{path}")
    print(f"GET {path:10s}  →  {r.status_code}  {r.json()}")

GET /           →  200  {'message': 'Mini Wikipedia RAG API'}
GET /version    →  200  {'version': '0.1.0'}


## 3. Available Routes (OpenAPI schema)

Pull the live OpenAPI spec and display every registered endpoint and its HTTP method.

In [11]:
schema = requests.get(f"{BASE_URL}/openapi.json").json()

print(f"API title   : {schema['info']['title']}")
print(f"API version : {schema['info']['version']}")
print()
print("Registered routes:")
for route_path, methods in schema["paths"].items():
    for method in methods:
        print(f"  {method.upper():6s}  {route_path}")

API title   : Mini Wikipedia RAG API
API version : 0.1.0

Registered routes:
  GET     /
  GET     /health
  GET     /version
  POST    /echo


## 4. Echo Endpoint

Simple request/response round-trip test — confirms the server can receive a JSON body and echo it back.

In [12]:
payload = {"message": "Hello from the notebook!"}
r = requests.post(f"{BASE_URL}/echo", json=payload)

print(f"Status code : {r.status_code}")
print(f"Sent        : {payload}")
print(f"Received    : {r.json()}")
assert r.json()["echo"] == payload["message"]
print("✓ Echo round-trip successful")

Status code : 200
Sent        : {'message': 'Hello from the notebook!'}
Received    : {'echo': 'Hello from the notebook!'}
✓ Echo round-trip successful


## 5. Auto-generated API Docs (/docs)

FastAPI auto-generates interactive Swagger UI docs. Confirm the `/docs` route returns an HTML page, then open it in a browser.

In [13]:
r = requests.get(f"{BASE_URL}/docs")
print(f"Status code  : {r.status_code}")
print(f"Content-Type : {r.headers['content-type']}")
assert r.status_code == 200
assert "text/html" in r.headers["content-type"]
print("✓ /docs is available")
print(f"\nOpen in browser: {BASE_URL}/docs")

Status code  : 200
Content-Type : text/html; charset=utf-8
✓ /docs is available

Open in browser: http://127.0.0.1:8000/docs


## 4. RAG Pipeline Overview

The three phases that will be added next, each backed by FastAPI endpoints and a dedicated notebook:

| Phase | What it does | Notebook |
|---|---|---|
| **Ingestion** | Load Wikipedia passages → embed → load into LlamaIndex VectorStoreIndex | `01_ingestion.ipynb` |
| **Query & Retrieval** | Embed a user query → similarity search → return top-K passages | `02_retrieval.ipynb` |
| **Augment & Generate** | Build augmented prompt → send to GPT-4o → return synthesized answer | `03_rag_pipeline.ipynb` |

```
User Query
   │
   ▼  embed (text-embedding-3-large)
Query Vector
   │
   ▼  similarity search (VectorStoreIndex)
Top-K Passages
   │
   ▼  augment prompt
[Context + Question] ──► GPT-4o ──► Answer
```